# moabb4 — Full Pipeline Benchmark (All train.py Pipelines)

Benchmarks **all 11 pipelines** from `train.py` on **8-channel** and **22-channel** configurations.
Protocol: within-subject 5-fold stratified CV.  Dataset: `BNCI2014_001`, 9 subjects.

| Family | Pipelines |
|---|---|
| Riemannian TS | cov_ts_lr, cov_ts_svm, cov_ts_mlp, cov_mdm |
| Augmented TS  | aug_ts_lr, aug_ts_svm, aug_ts_mlp |
| CSP           | csp_svm, csp_lda, csp_rf, csp_mlp |

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import mne
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from IPython.display import display

import moabb
from moabb.datasets import BNCI2014_001
from moabb.paradigms import LeftRightImagery
moabb.set_log_level('info')

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from mne.decoding import CSP
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM

# ── Paths — notebook lives in src/notebooks/, outputs go to PythonBCI/models ─
# Do NOT override MNE_DATA — reuse the same cache as moabb3_apples_to_apples
NB_DIR    = Path.cwd()  # src/notebooks/ when run from VS Code
MODEL_DIR = NB_DIR.parent.parent / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Outputs: {MODEL_DIR}')
print(f'MNE data: {mne.get_config("MNE_DATA", "(default)")}' )

# ── Global plot aesthetics ─────────────────────────────────────────────────────
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 18,
})
sns.set_style('whitegrid')
SEED = 42
np.random.seed(SEED)
print('Setup complete.')


## 1. Pipeline Definitions (mirrors train.py exactly)

In [ ]:
class AugmentedDataset(BaseEstimator, TransformerMixin):
    """Delay-embedding augmentation — identical to train.py."""
    def __init__(self, order=2, lag=1):
        self.order = order; self.lag = lag
    def fit(self, X, y=None): return self
    def transform(self, X):
        if self.order <= 1: return X
        chunks = []
        for p in range(self.order):
            s = p * self.lag
            e = X.shape[2] - (self.order - 1 - p) * self.lag
            chunks.append(X[:, :, s:e])
        return np.concatenate(chunks, axis=1)


def make_pipelines():
    LR  = lambda C=1.0: LogisticRegression(C=C, max_iter=2000, solver='lbfgs', random_state=SEED)
    SVM = lambda C=1.0, k='rbf': SVC(C=C, kernel=k, probability=True, random_state=SEED)
    MLP = lambda h=(100,): MLPClassifier(hidden_layer_sizes=h, max_iter=2000, random_state=SEED)
    CSP4 = lambda: CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=False)
    COV  = lambda e='oas': Covariances(estimator=e)
    TS   = lambda: TangentSpace(metric='riemann')
    SC   = StandardScaler
    return {
        # Riemannian Tangent Space
        'cov_ts_lr':  Pipeline([('cov',COV()),('ts',TS()),('sc',SC()),('clf',LR())]),
        'cov_ts_svm': Pipeline([('cov',COV()),('ts',TS()),('sc',SC()),('clf',SVM())]),
        'cov_ts_mlp': Pipeline([('cov',COV()),('ts',TS()),('sc',SC()),('clf',MLP())]),
        'cov_mdm':    Pipeline([('cov',COV()),('clf',MDM(metric='riemann'))]),
        # Augmented TS (delay embedding)
        'aug_ts_lr':  Pipeline([('aug',AugmentedDataset(2)),('cov',COV('lwf')),('ts',TS()),('sc',SC()),('clf',LR(0.1))]),
        'aug_ts_svm': Pipeline([('aug',AugmentedDataset(2)),('cov',COV()),('ts',TS()),('sc',SC()),('clf',SVM())]),
        'aug_ts_mlp': Pipeline([('aug',AugmentedDataset(2)),('cov',COV()),('ts',TS()),('sc',SC()),('clf',MLP())]),
        # CSP
        'csp_svm':    Pipeline([('csp',CSP4()),('clf',SVM())]),
        'csp_lda':    Pipeline([('csp',CSP4()),('clf',LDA())]),
        'csp_rf':     Pipeline([('csp',CSP4()),('clf',RandomForestClassifier(100,random_state=SEED))]),
        'csp_mlp':    Pipeline([('csp',CSP4()),('clf',MLP())]),
    }

PIPELINES  = make_pipelines()
PIPE_ORDER = list(PIPELINES.keys())
FAMILY = {
    'cov_ts_lr':'Riemannian','cov_ts_svm':'Riemannian',
    'cov_ts_mlp':'Riemannian','cov_mdm':'Riemannian',
    'aug_ts_lr':'Aug-TS','aug_ts_svm':'Aug-TS','aug_ts_mlp':'Aug-TS',
    'csp_svm':'CSP','csp_lda':'CSP','csp_rf':'CSP','csp_mlp':'CSP',
}
print(f'{len(PIPELINES)} pipelines ready: {PIPE_ORDER}')


## 2. Data Extraction
Uses the same default MOABB cache as `moabb3_apples_to_apples` — no re-download needed.

In [ ]:
UHB_CHANNELS = ['FC3','C3','CP3','Cz','CPz','FC4','C4','CP4']
SFREQ        = 250
DATASET_TAG  = 'BNCI2014001'

dataset      = BNCI2014_001()
SUBJECTS     = dataset.subject_list
print('Subjects:', SUBJECTS)

paradigm_8ch  = LeftRightImagery(channels=UHB_CHANNELS, resample=SFREQ)
paradigm_22ch = LeftRightImagery(resample=SFREQ)

print('Extracting 8-channel data...')
X_8,  y_8,  meta_8  = paradigm_8ch.get_data(dataset=dataset, subjects=SUBJECTS)
print('Extracting 22-channel data...')
X_22, y_22, meta_22 = paradigm_22ch.get_data(dataset=dataset, subjects=SUBJECTS)

le       = LabelEncoder()
y_8_num  = le.fit_transform(y_8)
y_22_num = le.fit_transform(y_22)

CONFIGS = {
    '8ch':  {'X': X_8,  'y': y_8_num,  'meta': meta_8},
    '22ch': {'X': X_22, 'y': y_22_num, 'meta': meta_22},
}
print(f'\n8ch  shape: {X_8.shape}')
print(f'22ch shape: {X_22.shape}')


## 3. Within-Subject 5-Fold CV
Results cached to `PythonBCI/models/BNCI2014001_moabb4_results.csv`.  Set `force_rerun=True` to re-evaluate.

In [ ]:
RESULTS_CSV  = MODEL_DIR / f'{DATASET_TAG}_moabb4_results.csv'
force_rerun  = False  # set True to ignore cache

def run_benchmark(configs, pipelines, subjects, n_splits=5):
    if RESULTS_CSV.exists() and not force_rerun:
        print(f'[CACHE] Loading {RESULTS_CSV.name}')
        return pd.read_csv(RESULTS_CSV)

    cv   = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    rows = []
    for ch_tag, d in configs.items():
        X, y, meta = d['X'], d['y'], d['meta']
        for pipe_name, pipeline in pipelines.items():
            print(f'  [{ch_tag}] {pipe_name}', flush=True)
            for subj in subjects:
                idx   = meta['subject'].astype(str) == str(subj)
                X_s, y_s = X[idx], y[idx]
                try:
                    cv_r = cross_validate(
                        clone(pipeline), X_s, y_s, cv=cv,
                        scoring=['accuracy','roc_auc'], n_jobs=-1
                    )
                    rows.append({
                        'pipeline': pipe_name, 'channels': ch_tag, 'subject': subj,
                        'accuracy': float(np.mean(cv_r['test_accuracy'])),
                        'roc_auc':  float(np.mean(cv_r['test_roc_auc'])),
                    })
                except Exception as exc:
                    print(f'    SKIP subj={subj}: {exc}')

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    print(f'Saved: {RESULTS_CSV}')
    return df

df_all   = run_benchmark(CONFIGS, PIPELINES, SUBJECTS)
df_means = (
    df_all.groupby(['pipeline','channels'])[['accuracy','roc_auc']]
    .mean().round(3).reset_index()
)
df_means['family'] = df_means['pipeline'].map(FAMILY)
print('\nGlobal means:')
display(df_means.sort_values('roc_auc', ascending=False).to_string(index=False))


## 4. Visualizations

In [ ]:
PALETTE  = {'8ch':'#4C72B0','22ch':'#DD8452'}
BG_COLOR = {'Riemannian':'#e8f4f8','Aug-TS':'#f8f0e8','CSP':'#f0f8e8'}

def add_family_shading(ax, pipe_order, family_map, y_pos=0.415):
    families = [family_map[p] for p in pipe_order]
    i, prev  = 0, families[0]
    for j, fam in enumerate(families + [None]):
        if fam != prev:
            ax.axvspan(i-.5, j-.5, alpha=0.20, color=BG_COLOR[prev], zorder=0)
            ax.text((i+j-1)/2, y_pos, prev, ha='center', fontsize=10,
                    color='#555', style='italic')
            i, prev = j, fam

def value_labels(ax, fontsize=9):
    for bar in ax.patches:
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x()+bar.get_width()/2, h+0.003,
                    f'{h:.2f}', ha='center', va='bottom',
                    fontsize=fontsize, fontweight='bold')

# ─── Main comparison: 8ch vs 22ch, all pipelines ─────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(20, 14), constrained_layout=True)
fig.suptitle('All train.py Pipelines — 8ch vs 22ch\nBNCI2014_001, 9 subjects, 5-fold CV',
             fontsize=18, fontweight='bold')

for ax, metric, title in zip(axes,
        ['roc_auc','accuracy'], ['Mean ROC AUC (higher = better)','Mean Accuracy']):
    sns.barplot(
        data=df_means, x='pipeline', y=metric, hue='channels',
        order=PIPE_ORDER, palette=PALETTE, ax=ax, width=0.65
    )
    ax.axhline(0.5, ls='--', color='red', lw=1.5, label='Chance (0.5)')
    ax.set_ylim(0.40, 1.0)
    ax.set_title(title, fontsize=16, pad=8)
    ax.set_xlabel('')
    ax.set_ylabel(title.split(' (')[0], fontsize=14)
    ax.tick_params(axis='x', rotation=35, labelsize=12)
    ax.legend(title='Channels', fontsize=12, title_fontsize=12)
    add_family_shading(ax, PIPE_ORDER, FAMILY)
    value_labels(ax)

plt.savefig(MODEL_DIR/f'{DATASET_TAG}_moabb4_pipeline_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── AUC Heatmap ──────────────────────────────────────────────────────────────
pivot = df_means.pivot(index='pipeline', columns='channels', values='roc_auc').loc[PIPE_ORDER]

fig, ax = plt.subplots(figsize=(7, 10))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.50, vmax=0.90, linewidths=0.6,
    annot_kws={'size':14,'weight':'bold'}, ax=ax
)
ax.set_title('ROC AUC — Pipeline × Channel Config', fontsize=16, fontweight='bold', pad=12)
ax.set_xlabel('Channel Config', fontsize=14)
ax.set_ylabel('Pipeline', fontsize=14)
ax.tick_params(labelsize=13)
plt.tight_layout()
plt.savefig(MODEL_DIR/f'{DATASET_TAG}_moabb4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Per-subject distribution — 8ch ───────────────────────────────────────────
df_8 = df_all[df_all['channels']=='8ch'].copy()

fig, ax = plt.subplots(figsize=(20, 7))
sns.boxplot(
    data=df_8, x='pipeline', y='roc_auc', order=PIPE_ORDER,
    palette='Set2', width=0.5, ax=ax, fliersize=3
)
sns.stripplot(
    data=df_8, x='pipeline', y='roc_auc', order=PIPE_ORDER,
    color='black', alpha=0.5, jitter=True, size=6, ax=ax
)
ax.axhline(0.5, ls='--', color='red', lw=1.5, label='Chance (0.5)')
ax.set_title('ROC AUC per Subject — 8ch (each dot = one subject mean)',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Pipeline', fontsize=14)
ax.set_ylabel('ROC AUC', fontsize=14)
ax.tick_params(axis='x', rotation=35, labelsize=12)
ax.legend(fontsize=12)
add_family_shading(ax, PIPE_ORDER, FAMILY, y_pos=0.41)
plt.tight_layout()
plt.savefig(MODEL_DIR/f'{DATASET_TAG}_moabb4_subject_dist.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── ΔAUC: 8ch minus 22ch ─────────────────────────────────────────────────────
d8  = df_means[df_means['channels']=='8ch'].set_index('pipeline')['roc_auc']
d22 = df_means[df_means['channels']=='22ch'].set_index('pipeline')['roc_auc']
delta = (d8-d22).loc[PIPE_ORDER].reset_index()
delta.columns = ['pipeline','delta_auc']
delta['color'] = delta['delta_auc'].apply(lambda x: '#2ecc71' if x>=0 else '#e74c3c')

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(delta['pipeline'], delta['delta_auc'],
              color=delta['color'], edgecolor='black', linewidth=0.7)
ax.axhline(0, color='black', lw=1)
ax.set_title('AUC Delta: 8ch − 22ch   (green = 8ch wins, red = 22ch wins)',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Pipeline', fontsize=14)
ax.set_ylabel('ΔAUC', fontsize=14)
ax.tick_params(axis='x', rotation=35, labelsize=12)
for bar, v in zip(bars, delta['delta_auc']):
    ax.text(bar.get_x()+bar.get_width()/2,
            v+(0.001 if v>=0 else -0.003),
            f'{v:+.3f}', ha='center',
            va='bottom' if v>=0 else 'top',
            fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR/f'{DATASET_TAG}_moabb4_channel_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print(delta.sort_values('delta_auc', ascending=False).to_string(index=False))


## 5. Summary Table

In [ ]:
best = df_means.sort_values('roc_auc', ascending=False).copy()
best = best[['family','pipeline','channels','roc_auc','accuracy']]
best.columns = ['Family','Pipeline','Channels','ROC AUC','Accuracy']
print('Top 10 configurations:')
display(best.head(10))

print('\nBest per family (8ch only):')
b8 = df_means[df_means['channels']=='8ch'].copy()
b8['family'] = b8['pipeline'].map(FAMILY)
display(b8.sort_values('roc_auc',ascending=False)
         .groupby('family')[['pipeline','roc_auc','accuracy']].first())

print(f'\nResults CSV : {RESULTS_CSV}')
print(f'Figures dir : {MODEL_DIR}')
